In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, LlamaConfig, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import math



In [ ]:
df = pd.read_csv('../data/whatsapp_chat_clean.csv')
df.head()

In [ ]:
#grouping the messages by hours for conversation, not perfect but good enough for now
eos = "<|endoftext|>"

df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.floor('h')

conversations =  (
    df.groupby(['username', 'hour'], as_index=False)
    .agg(message=('message', lambda s: f"{eos}".join(s.astype(str))))
)

In [ ]:
conversations

In [ ]:
#adding the username to the message to help with training
conversations['message'] = conversations['username'] + ': ' + conversations['message']

In [ ]:
conversations.head()

In [ ]:
#splitting the data into training and validation sets
def split_user(g, val_frac=0.1):
    g = g.sort_values('hour')
    n_val = max(1, int(round(len(g) * val_frac)))  # at least 1 val hour if possible
    if len(g) < 2:
        # too few hours to split put everything in train
        g = g.copy()
        g['split'] = 'train'
        return g
    g = g.copy()
    g['split'] = ['train'] * (len(g) - n_val) + ['val'] * n_val
    return g

split = conversations.groupby('username', group_keys=False).apply(split_user)

train_df = split[split['split'] == 'train'].drop(columns='split')
val_df   = split[split['split'] == 'val'].drop(columns='split')

In [ ]:
train_df

In [ ]:
val_df

In [ ]:
train_df.head()

In [ ]:
#loading the base model for our custom tokenizer
base = AutoTokenizer.from_pretrained('gpt2')

In [ ]:
#creating an iterator for the training data, it needs to be a batch of strings
iterator = train_df['message'].to_list()

In [ ]:
tokenizer = base.train_new_from_iterator(iterator, vocab_size=8000)
tokenizer.save_pretrained("../models/chat-tokenizer")

In [ ]:
#loading the tokenizer and testing it on a sequence
tokenizer = AutoTokenizer.from_pretrained("../models/chat-tokenizer")
sequence = "Adam: whats up guys!"
tokens = tokenizer.tokenize(sequence)
print(tokens)

In [ ]:
config = LlamaConfig(
    vocab_size=tokenizer.vocab_size, 
    hidden_size=256,
    intermediate_size=1024,             
    num_hidden_layers=4,
    num_attention_heads=4,             
    max_position_embeddings=256,        
    bos_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
config

In [ ]:
model = AutoModelForCausalLM.from_config(config)

In [ ]:
model.num_parameters()

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
#training the model
train_dataset = Dataset.from_pandas(train_df[['message']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[['message']].reset_index(drop=True))

In [ ]:
block_size = 256

def tokenize(batch):
    out = tokenizer(batch['message'], add_special_tokens=False)
    return {'input_ids': out['input_ids']}  # dropping the attention_mask

def group_texts(examples):
    concatenated = sum(examples['input_ids'], [])
    total_len = (len(concatenated) // block_size) * block_size
    concatenated = concatenated[:total_len]
    chunks = [concatenated[i:i+block_size] for i in range(0, total_len, block_size)]
    return {'input_ids': chunks, 'labels': chunks}

tokenized_train = train_dataset.map(tokenize,
                                    batched=True,
                                    remove_columns=train_dataset.column_names
                                    )
tokenized_val   = val_dataset.map(tokenize,
                                  batched=True,
                                  remove_columns=val_dataset.column_names
                                  )

lm_train = tokenized_train.map(group_texts, batched=True)
lm_val   = tokenized_val.map(group_texts, batched=True)

In [ ]:
#training the model
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
args = TrainingArguments(
    output_dir='../models/chat-llama-tiny',
    num_train_epochs=20,          # random init needs many passes
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,           # higher than fine-tuning
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)
trainer.train()

In [ ]:
#testing the perplexity of the model
metrics = trainer.evaluate()
ppl = math.exp(metrics['eval_loss'])
print(f"random baseline ~ {tokenizer.vocab_size}")
print(f"mine: {ppl:.1f}")

In [ ]:
#testing the model

model.eval()
prompt = "Adam"
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

out = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(out[0], skip_special_tokens=False))